# Implementación de Top-k Random Forest Forecasts
## Predicción de Duración de Viajes EcoBici Guadalajara

## Kaleb Aguilar  

**Artículo base:** Koster, N. & Krüger, F. — *Simplifying Random Forests' Probabilistic Forecasts*, The American Statistician, 2025.


## 0. Imports

In [102]:
import numpy as np
import csv
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

## 1. Carga de datos

In [103]:
df = pd.read_csv("datos_abiertos_2026_02_clean.csv")
print(df.columns.tolist())


df = df.dropna()
print(f'Shape: {df.shape}')
df

['Viaje_Id', 'Usuario_Id', 'Genero', 'Año_de_nacimiento', 'Inicio_del_viaje', 'Fin_del_viaje', 'Origen_Id', 'Destino_Id']
Shape: (331315, 8)


,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
0,42755487,530602,M,1995.0,2026-02-01 00:00:01,2026-02-01 00:10:58,50,13
1,42755488,71721,M,1987.0,2026-02-01 00:00:12,2026-02-01 00:02:02,57,260
2,42755489,2211330,F,2003.0,2026-02-01 00:00:15,2026-02-01 00:19:56,65,86
3,42755490,4647016,M,2003.0,2026-02-01 00:00:48,2026-02-01 00:08:37,48,36
4,42755491,71721,M,1987.0,2026-02-01 00:02:24,2026-02-01 00:08:38,260,50
...,...,...,...,...,...,...,...,...
333751,43136888,4542423,M,1995.0,2026-02-28 23:56:57,2026-03-01 00:21:24,203,328
333752,43136889,2344334,M,2004.0,2026-02-28 23:57:20,2026-03-01 00:23:26,87,383
333753,43136890,42747,M,1989.0,2026-02-28 23:57:24,2026-03-01 00:02:38,49,84
333754,43136891,2106426,F,1987.0,2026-02-28 23:57:34,2026-03-01 00:02:38,49,84


## 2. Preprocesamiento


In [104]:
df['Inicio_del_viaje'] = pd.to_datetime(df['Inicio_del_viaje'])
df['Fin_del_viaje']    = pd.to_datetime(df['Fin_del_viaje'])

df['duracion_del_viaje'] = (df['Fin_del_viaje'] - df['Inicio_del_viaje']).dt.total_seconds() / 60
df

,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id,duracion_del_viaje
0,42755487,530602,M,1995.0,2026-02-01 00:00:01,2026-02-01 00:10:58,50,13,10.950000
1,42755488,71721,M,1987.0,2026-02-01 00:00:12,2026-02-01 00:02:02,57,260,1.833333
2,42755489,2211330,F,2003.0,2026-02-01 00:00:15,2026-02-01 00:19:56,65,86,19.683333
3,42755490,4647016,M,2003.0,2026-02-01 00:00:48,2026-02-01 00:08:37,48,36,7.816667
4,42755491,71721,M,1987.0,2026-02-01 00:02:24,2026-02-01 00:08:38,260,50,6.233333
...,...,...,...,...,...,...,...,...,...
333751,43136888,4542423,M,1995.0,2026-02-28 23:56:57,2026-03-01 00:21:24,203,328,24.450000
333752,43136889,2344334,M,2004.0,2026-02-28 23:57:20,2026-03-01 00:23:26,87,383,26.100000
333753,43136890,42747,M,1989.0,2026-02-28 23:57:24,2026-03-01 00:02:38,49,84,5.233333
333754,43136891,2106426,F,1987.0,2026-02-28 23:57:34,2026-03-01 00:02:38,49,84,5.066667


### 2.2 Filtrado de registros inválidos y outliers

In [105]:
df = df[df['duracion_del_viaje'] > 0].copy()
df = df.dropna(subset=['duracion_del_viaje'])

p99 = df['duracion_del_viaje'].quantile(0.99)
df  = df[df['duracion_del_viaje'] <= min(p99, 240)].copy()

print(f'Registros tras limpieza: {len(df):,}')

Registros tras limpieza: 326,176


### 2.3 Variables extra

Aquí estoy agregando los

In [106]:
df['edad']          = pd.Timestamp.now().year - df['Año_de_nacimiento']
df['hora_inicio']   = df['Inicio_del_viaje'].dt.hour
df['dia_semana']    = df['Inicio_del_viaje'].dt.dayofweek
df['es_fin_semana'] = (df['dia_semana'] >= 5).astype(int)
df['Genero'] = df['Genero'].str.strip().map({'M': 0, 'F': 1})
df = df.dropna(subset=['Genero'])
df

,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id,duracion_del_viaje,edad,hora_inicio,dia_semana,es_fin_semana
0,42755487,530602,0,1995.0,2026-02-01 00:00:01,2026-02-01 00:10:58,50,13,10.950000,31.0,0,6,1
1,42755488,71721,0,1987.0,2026-02-01 00:00:12,2026-02-01 00:02:02,57,260,1.833333,39.0,0,6,1
2,42755489,2211330,1,2003.0,2026-02-01 00:00:15,2026-02-01 00:19:56,65,86,19.683333,23.0,0,6,1
3,42755490,4647016,0,2003.0,2026-02-01 00:00:48,2026-02-01 00:08:37,48,36,7.816667,23.0,0,6,1
4,42755491,71721,0,1987.0,2026-02-01 00:02:24,2026-02-01 00:08:38,260,50,6.233333,39.0,0,6,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
333751,43136888,4542423,0,1995.0,2026-02-28 23:56:57,2026-03-01 00:21:24,203,328,24.450000,31.0,23,5,1
333752,43136889,2344334,0,2004.0,2026-02-28 23:57:20,2026-03-01 00:23:26,87,383,26.100000,22.0,23,5,1
333753,43136890,42747,0,1989.0,2026-02-28 23:57:24,2026-03-01 00:02:38,49,84,5.233333,37.0,23,5,1
333754,43136891,2106426,1,1987.0,2026-02-28 23:57:34,2026-03-01 00:02:38,49,84,5.066667,39.0,23,5,1


### 2.4 Codificación y construcción de X, y

In [107]:
TARGET = 'duracion_del_viaje'
DROP   = ['Viaje_Id', 'Usuario_Id', 'Inicio_del_viaje', 'Fin_del_viaje','Año_de_nacimiento', TARGET]

feature_cols = [c for c in df.columns if c not in DROP]
X = df[feature_cols].values.astype(float)
y = df[TARGET].values.astype(float)

print(f'Features: {feature_cols}')
X

Features: ['Genero', 'Origen_Id', 'Destino_Id', 'edad', 'hora_inicio', 'dia_semana', 'es_fin_semana']


array([[  0.,  50.,  13., ...,   0.,   6.,   1.],
       [  0.,  57., 260., ...,   0.,   6.,   1.],
       [  1.,  65.,  86., ...,   0.,   6.,   1.],
       ...,
       [  0.,  49.,  84., ...,  23.,   5.,   1.],
       [  1.,  49.,  84., ...,  23.,   5.,   1.],
       [  0.,  66.,  72., ...,  23.,   5.,   1.]])

### 2.5 División train / test

In [108]:
# Ordenar por tiempo
df = df.sort_values('Inicio_del_viaje').reset_index(drop=True)

# Split 70/30 respetando el orden cronológico
split_idx = int(len(df) * 0.70)

X_train = X[:split_idx]
y_train = y[:split_idx]
X_test  = X[split_idx:]
y_test  = y[split_idx:]

print(f'Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}')
print(f'Train hasta: {df["Inicio_del_viaje"].iloc[split_idx-1]}')
print(f'Test desde:  {df["Inicio_del_viaje"].iloc[split_idx]}')

Train: 228,323  |  Test: 97,853
Train hasta: 2026-02-19 08:57:07
Test desde:  2026-02-19 08:57:16


## 3. Random Forest

Cada árbol se entrena sobre un bootstrap del conjunto de entrenamiento.
La predicción final es el promedio de todos los árboles.

In [109]:
N_ESTIMATORS = 1000
MAX_FEATURES = len(feature_cols)

trees        = []   # lista de árboles entrenados
bootstrap_idx = []  # índices bootstrap de cada árbol

n_train    = X_train.shape[0]
n_features = X_train.shape[1]
max_feat   = max(1, int(np.sqrt(n_features)))

for i in range(N_ESTIMATORS):
    # Bootstrap
    idx = np.random.choice(n_train, size=n_train, replace=True)
    X_b, y_b = X_train[idx], y_train[idx]

    # Subconjunto de features
    feat_idx = np.random.choice(n_features, size=max_feat, replace=False)

    tree = DecisionTreeRegressor(random_state=i)
    tree.fit(X_b[:, feat_idx], y_b)

    trees.append((tree, feat_idx))
    bootstrap_idx.append(idx)

    if (i + 1) % 200 == 0:
        print(f'  {i+1}/{N_ESTIMATORS} árboles entrenados')

print('RF listo.')

  200/1000 árboles entrenados
  400/1000 árboles entrenados
  600/1000 árboles entrenados
  800/1000 árboles entrenados
  1000/1000 árboles entrenados
RF listo.


### 3.1 Predicción del RF (promedio de árboles)

In [110]:
def rf_predict(trees, X):
    preds = np.zeros((len(trees), len(X)))
    for i, (tree, feat_idx) in enumerate(trees):
        preds[i] = tree.predict(X[:, feat_idx])
    return preds.mean(axis=0)

def metrics(y_true, y_pred, label=''):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f'[{label}]  MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.4f}')
    return {'label': label, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

y_pred_rf = rf_predict(trees, X_test)
results = [metrics(y_test, y_pred_rf, 'RF completo')]

[RF completo]  MAE=4.994  RMSE=6.178  R²=0.0964


## 4. Visualizaciones de regresión

### 4.1 Distribución del target (train vs test)

In [111]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=y_train, name='Train', opacity=0.6,
                            nbinsx=60, marker_color='#0077b6'))
fig.add_trace(go.Histogram(x=y_test,  name='Test',  opacity=0.6,
                            nbinsx=60, marker_color='#48cae4'))
fig.update_layout(
    barmode='overlay',
    title='Distribución del target — duración (min)',
    xaxis_title='Duración (min)', yaxis_title='Frecuencia'
)
fig.show()

### 4.2 Predicciones vs valores reales — RF completo

In [112]:
sample = np.random.choice(len(y_test), size=min(2000, len(y_test)), replace=False)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=y_test[sample], y=y_pred_rf[sample],
    mode='markers', marker=dict(color='#0096c7', opacity=0.4, size=4),
    name='RF completo'
))
lim = [0, y_test.max()]
fig.add_trace(go.Scatter(x=lim, y=lim, mode='lines',
                          line=dict(color='red', dash='dash'), name='Ideal'))
fig.update_layout(
    title='Predicciones vs valores reales — RF completo',
    xaxis_title='Real (min)', yaxis_title='Predicción (min)'
)
fig.show()

## 5. Cálculo de pesos RF mediante coincidencia de hojas

Para cada observación de prueba $x_j$, el peso $w_i$ asignado a la observación de
entrenamiento $x_i$ es la fracción de árboles en los que ambas caen en la misma hoja:

$$w_i(x_j) = \frac{1}{T} \sum_{t=1}^{T} \frac{\mathbf{1}[\ell_t(x_j) = \ell_t(x_i)]}{|\text{hoja}_t(x_j)|}$$

In [120]:
#  N_DEMO: observaciones de test para hacer manejable el cálculo

N_DEMO = 500
X_demo = X_test[:N_DEMO]
y_demo = y_test[:N_DEMO]

def compute_rf_weights(trees, X_train, X_demo):
    n_test  = X_demo.shape[0]
    n_train = X_train.shape[0]
    W = np.zeros((n_test, n_train), dtype=np.float32)

    for tree, feat_idx in trees:
        leaves_train = tree.apply(X_train[:, feat_idx])   # (n_train,)
        leaves_test  = tree.apply(X_demo[:, feat_idx])    # (n_test,)

        match = (leaves_test[:, None] == leaves_train[None, :])  # (n_test, n_train)
        leaf_sizes = match.sum(axis=1, keepdims=True)
        leaf_sizes[leaf_sizes == 0] = 1
        W += match.astype(np.float32) / leaf_sizes

    W /= len(trees)
    return W

print('Calculando pesos RF...')
W = compute_rf_weights(trees, X_train, X_demo)
print(f'Matriz de pesos: {W.shape}')

Calculando pesos RF...
Matriz de pesos: (500, 228323)


### 5.1 Verificación: reconstrucción con pesos completos

In [121]:
y_pred_weighted = W @ y_train
metrics(y_demo, y_pred_weighted, 'Pesos completos (demo)')
print('(Debe ser cercano al RF clásico sobre el mismo subconjunto)')

[Pesos completos (demo)]  MAE=5.073  RMSE=6.212  R²=0.1097
(Debe ser cercano al RF clásico sobre el mismo subconjunto)


In [122]:
W

array([[0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 2.3674831e-05,
        2.3674831e-05, 1.5906490e-04],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 2.3674831e-05,
        2.3674831e-05, 2.3674831e-05],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 2.3674831e-05,
        2.3674831e-05, 2.3674831e-05],
       ...,
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 3.6560991e-06,
        3.6560991e-06, 3.6560991e-06],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 3.6560991e-06,
        3.6560991e-06, 3.6560991e-06],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 3.6560991e-06,
        3.6560991e-06, 3.6560991e-06]], dtype=float32)

## 6. Implementación Top-k

1. Ordenar pesos de mayor a menor.
2. Conservar solo los $k$ más grandes.
3. Renormalizar: $\sum w_i = 1$.
4. Recalcular $\hat{y} = W_{\text{top-k}} \, y_{\text{train}}$.

In [123]:
def topk_predict(W, y_train, k):
    W_k = W.copy()

    # threshold: tratar valores muy pequeños como cero
    W_k[W_k < 1e-5] = 0.0

    # renormalizar antes de seleccionar top-k
    row_sums = W_k.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    W_k /= row_sums

    # ahora sí seleccionar top-k
    top_idx = np.argpartition(W_k, -k, axis=1)[:, -k:]
    mask = np.zeros_like(W_k, dtype=bool)
    for j in range(W_k.shape[0]):
        mask[j, top_idx[j]] = True
    W_k[~mask] = 0.0

    # renormalizar final
    row_sums = W_k.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    W_k /= row_sums

    return W_k @ y_train, W_k

K_VALUES = [3, 5, 10]
topk_res = {}

for k in K_VALUES:
    y_k, W_k = topk_predict(W, y_train, k)
    m = metrics(y_demo, y_k, f'Top-{k}')
    topk_res[k] = {'metrics': m, 'W': W_k, 'y_pred': y_k}

[Top-3]  MAE=2.756  RMSE=4.498  R²=0.5333
[Top-5]  MAE=2.774  RMSE=4.390  R²=0.5554
[Top-10]  MAE=2.994  RMSE=4.505  R²=0.5317


## 7. Visualizaciones comparativas

### 7.1 Predicciones vs real: RF completo vs Top-k

In [124]:
colors = {'RF completo': '#023e8a', 'Top-3': '#023e8a',
          'Top-5': '#023e8a', 'Top-10': '#023e8a'}

s = np.random.choice(N_DEMO, size=min(500, N_DEMO), replace=False)

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=['RF completo', 'Top-3', 'Top-5', 'Top-10'])

pares = [
    ('RF completo', y_pred_rf[:N_DEMO], 1, 1),
    ('Top-3',  topk_res[3]['y_pred'],  1, 2),
    ('Top-5',  topk_res[5]['y_pred'],  2, 1),
    ('Top-10', topk_res[10]['y_pred'], 2, 2),
]

for label, yp, row, col in pares:
    fig.add_trace(go.Scatter(
        x=y_demo[s], y=yp[s],
        mode='markers',
        marker=dict(color=colors[label], opacity=0.5, size=4),
        name=label
    ), row=row, col=col)
    lim = [0, y_demo.max()]
    fig.add_trace(go.Scatter(
        x=lim, y=lim, mode='lines',
        line=dict(color='red', dash='dash'), showlegend=False
    ), row=row, col=col)

fig.update_layout(title='Predicciones vs real (min) — RF completo y Top-k', height=700)
fig.show()

### 7.2 Distribución de pesos — una observación de prueba

In [127]:
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=['RF completo', 'Top-3', 'Top-5', 'Top-10'])

configs = [
    ('RF completo', y_test[:N_DEMO] - y_pred_rf[:N_DEMO], '#023e8a', 1, 1),
    ('Top-3',  y_demo - topk_res[3]['y_pred'],  '#0096c7', 1, 2),
    ('Top-5',  y_demo - topk_res[5]['y_pred'],  '#00b4d8', 2, 1),
    ('Top-10', y_demo - topk_res[10]['y_pred'], '#48cae4', 2, 2),
]

for label, residuos, color, row, col in configs:
    fig.add_trace(go.Histogram(
        x=residuos, nbinsx=80,
        marker_color=color, opacity=0.75,
        name=label, showlegend=False
    ), row=row, col=col)
    fig.add_vline(x=0, line_dash='dash', line_color='red', row=row, col=col)

fig.update_layout(
    title='Distribución de residuos — RF completo vs Top-k',
    height=600
)
fig.update_xaxes(title_text='Residuo (min)')
fig.update_yaxes(title_text='Frecuencia')
fig.show()